# CA1 — Building a Categorical Naive Bayes Classifier

The target columns are `poisonous` for Mushroom Dataset and `class` for Soybean Dataset.


**Name:** Alireza Mohammadzade  
**Student ID:** 810104490


## Part 2 — Manual Naive Bayes computation

The table in the assignment has 21 rows. Rows 1 to 20 are the training data;
row 21 is a new sample to classify. The columns are `bruises`, `gill-size`, and
`poisonous`.

The training rows are:

| row | bruises | gill-size | poisonous |
|-----|---------|-----------|-----------|
| 1   | f       | b         | p         |
| 2   | t       | b         | e         |
| 3   | f       | b         | e         |
| 4   | f       | n         | p         |
| 5   | f       | n         | p         |
| 6   | t       | b         | e         |
| 7   | t       | b         | e         |
| 8   | t       | b         | p         |
| 9   | t       | b         | e         |
| 10  | t       | b         | e         |
| 11  | f       | n         | e         |
| 12  | f       | n         | p         |
| 13  | f       | b         | e         |
| 14  | t       | b         | p         |
| 15  | f       | b         | e         |
| 16  | t       | n         | e         |
| 17  | t       | b         | e         |
| 18  | t       | b         | e         |
| 19  | f       | b         | p         |
| 20  | f       | n         | p         |

The new sample (row 21) is `bruises = f`, `gill-size = b`.

### (a) Class counts and prior probabilities

Count the training rows of each class.

- Class `p` (poisonous): rows 1, 4, 5, 8, 12, 14, 19, 20. Count = 8.
- Class `e` (edible):    rows 2, 3, 6, 7, 9, 10, 11, 13, 15, 16, 17, 18. Count = 12.

Total training rows = 20. Priors:

$$P(C = p) = \frac{8}{20} = 0.40, \qquad P(C = e) = \frac{12}{20} = 0.60.$$

### (b) Conditional probabilities

For every class, count the training rows of that class in which `bruises = f`
and in which `gill-size = b`.

Class `p` (8 rows: 1, 4, 5, 8, 12, 14, 19, 20):

- `bruises = f`: rows 1, 4, 5, 12, 19, 20. Count = 6.
  $$P(\text{bruises} = f \mid p) = \frac{6}{8} = 0.75.$$
- `gill-size = b`: rows 1, 8, 14, 19. Count = 4.
  $$P(\text{gill-size} = b \mid p) = \frac{4}{8} = 0.50.$$

Class `e` (12 rows: 2, 3, 6, 7, 9, 10, 11, 13, 15, 16, 17, 18):

- `bruises = f`: rows 3, 11, 13, 15. Count = 4.
  $$P(\text{bruises} = f \mid e) = \frac{4}{12} = \frac{1}{3} \approx 0.3333.$$
- `gill-size = b`: rows 2, 3, 6, 7, 9, 10, 13, 15, 17, 18. Count = 10.
  $$P(\text{gill-size} = b \mid e) = \frac{10}{12} = \frac{5}{6} \approx 0.8333.$$

Summary:

| conditional probability       | class `p` | class `e` |
|-------------------------------|-----------|-----------|
| `P(bruises = f \| C)`          | 6/8  = 0.75      | 4/12 = 1/3  ≈ 0.3333 |
| `P(gill-size = b \| C)`        | 4/8  = 0.50      | 10/12 = 5/6 ≈ 0.8333 |

### (c) Class scores for row 21

Using the independence assumption,

$$S(C) = P(C)\, P(\text{bruises} = f \mid C)\, P(\text{gill-size} = b \mid C).$$

For class `p`:

$$S(p) = 0.40 \times 0.75 \times 0.50 = 0.15.$$

For class `e`:

$$S(e) = 0.60 \times \frac{1}{3} \times \frac{5}{6}
       = \frac{3}{5} \times \frac{1}{3} \times \frac{5}{6}
       = \frac{1}{6} \approx 0.1667.$$

Comparison:

$$S(e) = 0.1667 > S(p) = 0.15.$$

The class with the larger score is `e`. The model therefore predicts
`poisonous = e` (edible) for row 21.

Remark: the true label of row 21 in the assignment table is `p`. On this single
sample the hand-computed model is therefore incorrect. This is expected at this
stage: the small dataset contains only 20 training rows, the model is unsmoothed
and uses only two features, and the two scores above are close (0.15 versus
0.1667). The purpose of Part 2 is the calculation itself; the correctness of the
classifier on the full test set is measured in Part 3.

## Part 3

In [1]:
import numpy as np
import pandas as pd

def load(name):
    return pd.read_csv(f"ca_datasets/{name}.csv", dtype=str, keep_default_na=False)

small_train = load("mushroom_small_train")
small_test  = load("mushroom_small_test")
full_train  = load("mushroom_full_train")
full_test   = load("mushroom_full_test")
soy_train   = load("soybean_train")
soy_test    = load("soybean_test")


FileNotFoundError: [Errno 2] No such file or directory: 'ca_datasets/mushroom_small_train.csv'

In [ ]:
TARGET_MUSHROOM = "poisonous"
SMALL_FEATURES = ["bruises", "gill-size"]

def fit_direct_nb(small_train, TARGET_MUSHROOM, SMALL_FEATURES):
    classes = sorted(small_train[TARGET_MUSHROOM].unique())
    priors = small_train[TARGET_MUSHROOM].value_counts(normalize=True).to_dict()

    cond_probs = {}
    for f in SMALL_FEATURES:
        cond_probs[f] = {}
        for c in classes:
            subset = small_train[small_train[TARGET_MUSHROOM] == c]
            cond_probs[f][c] = subset[f].value_counts(normalize=True).to_dict()

    return {
        "target": TARGET_MUSHROOM,
        "features": list(SMALL_FEATURES),
        "classes": classes,
        "priors": priors,
        "cond_probs": cond_probs,
    }

def predict_direct_nb(model, X):
    all_scores = []
    for _, row in X.iterrows():
        scores = {}
        for c in model["classes"]:
            s = model["priors"].get(c, 0.0)
            for f in model["features"]:
                s *= model["cond_probs"][f][c].get(row[f], 0.0)
            scores[c] = s
        all_scores.append(scores)
    preds = []
    for scores in all_scores:
        best_class = max(
            model["classes"],
            key=lambda c: (scores[c], model["priors"].get(c, 0.0))
        )
        preds.append(best_class)
    return preds

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

small_model_direct = fit_direct_nb(small_train, TARGET_MUSHROOM, SMALL_FEATURES)
small_predictions_direct = predict_direct_nb(
    small_model_direct, small_test[SMALL_FEATURES]
)
print(f"Small Mushroom baseline accuracy: "
      f"{accuracy(small_test[TARGET_MUSHROOM], small_predictions_direct):.2%}")

Small Mushroom baseline accuracy: 76.80%


In [ ]:
full_features = [c for c in full_train.columns if c != TARGET_MUSHROOM]

full_model_direct_p3 = fit_direct_nb(full_train, TARGET_MUSHROOM, full_features)
full_predictions_direct_p3 = predict_direct_nb(full_model_direct_p3, full_test[full_features])

acc_small = accuracy(small_test[TARGET_MUSHROOM], small_predictions_direct)
acc_full  = accuracy(full_test[TARGET_MUSHROOM],  full_predictions_direct_p3)

print(f"{'dataset':<18} | {'#features':>9} | {'accuracy':>9}")
print("-" * 44)
print(f"{'Mushroom (small)':<18} | {len(SMALL_FEATURES):>9} | {acc_small:>9.2%}")
print(f"{'Mushroom (full)':<18} | {len(full_features):>9} | {acc_full:>9.2%}")

def n_cond_probs(train, target, features):
    n_classes = train[target].nunique()
    per_class = sum(train[f].nunique() for f in features)
    joint = 1
    for f in features:
        joint *= train[f].nunique()                           
    return n_classes, per_class, n_classes * per_class, n_classes * joint

print()
print(f"{'dataset':<18} | {'classes':>7} | {'sum K_i':>7} | {'Naive Bayes: C*sum K_i':>22} | {'full joint: C*prod K_i':>24}")
print("-" * 90)
for name, tr, feats in [("Mushroom (small)", small_train, SMALL_FEATURES),
                        ("Mushroom (full)",  full_train,  full_features)]:
    c, s, nb_params, joint_params = n_cond_probs(tr, TARGET_MUSHROOM, feats)
    print(f"{name:<18} | {c:>7} | {s:>7} | {nb_params:>22,} | {joint_params:>24,}")

dataset            | #features |  accuracy
--------------------------------------------
Mushroom (small)   |         2 |    76.80%
Mushroom (full)    |        22 |    99.45%

dataset            | classes | sum K_i | Naive Bayes: C*sum K_i |   full joint: C*prod K_i
------------------------------------------------------------------------------------------
Mushroom (small)   |       2 |       4 |                      8 |                        8
Mushroom (full)    |       2 |     117 |                    234 |      243,799,621,632,000


### Part 3 — Analysis

**1. Is the accuracy the same on the small and the full dataset?**
No: **76.80 %** with the two features of the small dataset versus **99.45 %** with all 22 features of the full dataset.
With only `bruises` and `gill-size` (both binary) every mushroom falls into one of just 4 feature combinations, and each combination contains both edible and poisonous mushrooms (e.g. `bruises = f, gill-size = b` has 1005 edible and 1082 poisonous training rows). Even the best possible rule built on these two features reaches only ≈ 76 % on the training data, so the small model is limited by the *information* in its features, not by the Naive Bayes assumption. The full dataset contains very informative features – `odor` alone classifies 98.6 % of the test mushrooms correctly – so the full model is far more accurate.

**2. How does using more features change the number of conditional probabilities?**
Naive Bayes needs $P(X_i = v \mid C)$ for every class, feature and value: $|C|\cdot\sum_i K_i$ numbers. This is **8** for the small dataset ($2\cdot 4$) and **234** for the full dataset ($2\cdot 117$), i.e. it grows *linearly* with the number of features. Without the independence assumption we would have to estimate the whole joint distribution $P(X_1,\dots,X_n \mid C)$, which needs $|C|\cdot\prod_i K_i \approx 2.4\cdot 10^{14}$ entries for the full dataset – far more than the 6 499 training rows. This is exactly why the independence assumption makes the problem tractable.

**3. Can we conclude that adding features always increases accuracy?**
No. The jump from 76.8 % to 99.45 % only shows that *these particular* extra features were very informative. Features that carry no information about the class add noise; features that are strongly correlated with ones already used are counted twice by the independence assumption, which makes the model over-confident; and every extra feature adds more probabilities to estimate, so with little data more zero counts appear (see the Soybean dataset in Part 5, with 35 features but only 234 rows). More features help only if they bring new information about the class and there is enough data to estimate them.

### Part 4

In [ ]:
def fit_nb(train, target, features, alpha=0.0):
    classes = sorted(train[target].unique())
    priors = train[target].value_counts(normalize=True).to_dict()

    value_domains = {}
    cond_probs = {}

    for f in features:
        train_values = set(train[f].dropna().unique())
        values = list(train_values) + ["__UNKNOWN__"]
        value_domains[f] = values

        cond_probs[f] = {}
        for c in classes:
            subset = train[train[target] == c]
            n_c = len(subset)
            counts = subset[f].value_counts().to_dict()
            K = len(values)

            cond_probs[f][c] = {}
            for v in values:
                if alpha == 0.0:
                    p = counts.get(v, 0) / n_c if n_c > 0 else 0.0
                else:
                    p = (counts.get(v, 0) + alpha) / (n_c + alpha * K)
                cond_probs[f][c][v] = p

    return {
        "target": target,
        "features": list(features),
        "classes": classes,
        "priors": priors,
        "value_domains": value_domains,
        "cond_probs": cond_probs,
        "alpha": alpha,
    }


def _feature_value(model, feature, value):
    if value in model["value_domains"][feature]:
        return value
    return "__UNKNOWN__"


def predict_nb_scores(model, X, use_log=False):
    all_scores = []

    for _, row in X.iterrows():
        scores = {}
        for c in model["classes"]:
            prior = model["priors"].get(c, 0.0)

            if use_log:
                s = np.log(prior) if prior > 0 else -np.inf
            else:
                s = prior

            for f in model["features"]:
                v = _feature_value(model, f, row[f])
                p = model["cond_probs"][f][c].get(v, 0.0)

                if use_log:
                    if p > 0:
                        s += np.log(p)
                    else:
                        s = -np.inf
                else:
                    s *= p

            scores[c] = s

        all_scores.append(scores)

    return all_scores


def predict_nb(model, X, use_log=False):
    preds = []
    for scores in predict_nb_scores(model, X, use_log=use_log):
        best_class = max(
            model["classes"],
            key=lambda c: (
                scores[c] if not (isinstance(scores[c], float) and np.isnan(scores[c])) else -np.inf,
                model["priors"].get(c, 0.0)
            )
        )
        preds.append(best_class)
    return preds


full_features = [c for c in full_train.columns if c != TARGET_MUSHROOM]

small_model_direct = fit_direct_nb(small_train, TARGET_MUSHROOM, SMALL_FEATURES)
small_pred_direct = predict_direct_nb(small_model_direct, small_test[SMALL_FEATURES])

small_model_log = fit_nb(small_train, TARGET_MUSHROOM, SMALL_FEATURES, alpha=0.0)
small_pred_log = predict_nb(small_model_log, small_test[SMALL_FEATURES], use_log=True)

print("Small Mushroom:")
print(f"  Direct accuracy: {accuracy(small_test[TARGET_MUSHROOM], small_pred_direct):.2%}")
print(f"  Log accuracy:    {accuracy(small_test[TARGET_MUSHROOM], small_pred_log):.2%}")

full_model_direct = fit_nb(full_train, TARGET_MUSHROOM, full_features, alpha=0.0)
full_pred_direct = predict_nb(full_model_direct, full_test[full_features], use_log=False)
full_scores_direct = predict_nb_scores(full_model_direct, full_test[full_features], use_log=False)

full_model_log = fit_nb(full_train, TARGET_MUSHROOM, full_features, alpha=0.0)
full_pred_log = predict_nb(full_model_log, full_test[full_features], use_log=True)

print("\nFull Mushroom:")
print(f"  Direct accuracy: {accuracy(full_test[TARGET_MUSHROOM], full_pred_direct):.2%}")
print(f"  Log accuracy:    {accuracy(full_test[TARGET_MUSHROOM], full_pred_log):.2%}")

nonzero_scores = [
    v for scores in full_scores_direct for v in scores.values() if v > 0
]
print("\nMinimum non-zero direct score on full Mushroom test:",
      min(nonzero_scores) if nonzero_scores else None)

In [ ]:
print("Small Mushroom - identical predictions (direct vs log):", small_pred_direct == small_pred_log)
print("Full  Mushroom - identical predictions (direct vs log):", full_pred_direct == full_pred_log)

def underflow_report(name, train, test, target, features, alpha):
    m = fit_nb(train, target, features, alpha=alpha)
    d = predict_nb_scores(m, test[features], use_log=False)
    l = predict_nb_scores(m, test[features], use_log=True)
    direct = np.array([[s[c] for c in m["classes"]] for s in d])
    logs   = np.array([[s[c] for c in m["classes"]] for s in l])
    n_underflow = int(((direct == 0) & np.isfinite(logs)).sum())
    return (name, alpha, direct.min(), logs.min(), n_underflow, direct.size)

rows = [
    underflow_report("Mushroom (small)", small_train, small_test, TARGET_MUSHROOM, SMALL_FEATURES, 1.0),
    underflow_report("Mushroom (full)",  full_train,  full_test,  TARGET_MUSHROOM, full_features, 1.0),
    underflow_report("Soybean",          soy_train,   soy_test,   "class",
                     [c for c in soy_train.columns if c != "class"], 0.01),
]
print(f"\nsmallest positive float64 (normal): {np.finfo(np.float64).tiny:.3e}")
print(f"{'dataset':<17} | {'alpha':>5} | {'min direct score':>16} | {'min log score':>13} | {'underflowed scores':>18}")
print("-" * 82)
for name, a, dmin, lmin, nu, tot in rows:
    print(f"{name:<17} | {a:>5} | {dmin:>16.3e} | {lmin:>13.2f} | {nu:>9} / {tot:<7}")


p = np.full(1000, 0.1)
print("\nSynthetic example, 1000 features with P(x_i | C) = 0.1:")
print("  product      :", np.prod(p), "  <- underflows to 0.0 (true value is 1e-1000)")
print("  sum of logs  :", np.log(p).sum(), "  <- still perfectly representable")

### Part 4 — Analysis

**1. Is the accuracy of the ordinary version equal to that of the logarithmic version?**
Yes. Both versions give **76.80 %** on the small and **99.45 %** on the full Mushroom dataset, and the predictions are identical sample by sample (checked above). The logarithm is strictly increasing, so $\arg\max_C S(C) = \arg\max_C \log S(C)$; taking logs changes the numbers but not which class has the largest score.

**2. Is any real Underflow observed?**
No, in neither dataset. To isolate underflow from genuine zero probabilities the check above uses $\alpha > 0$ (so no conditional probability is 0): then a direct score equal to `0.0` could only come from underflow, and there are **0** such scores (0 / 3 250 on the small and the full Mushroom data, 0 / 1 387 on Soybean). The smallest direct scores are $8.0\cdot10^{-32}$ (full Mushroom) and $1.3\cdot10^{-107}$ (Soybean), still very far above the smallest representable `float64` ($2.2\cdot10^{-308}$); the corresponding log-scores are only $-71.6$ and $-246.1$. With 22 or 35 features the products simply do not get small enough. Underflow becomes real only with hundreds of features, as the synthetic example shows (1000 factors of 0.1 give `0.0` directly, but $-2302.6$ in log-space). The exact zeros that do occur on Soybean without smoothing are *not* underflow – they come from conditional probabilities that are exactly 0 (Part 5). So the logarithmic version is not needed for correctness on these datasets, but it is the safe choice in general.

## Part 5

In [ ]:
SOY_TARGET = "class"
soy_features = [c for c in soy_train.columns if c != SOY_TARGET]

soy_model_direct = fit_nb(soy_train, SOY_TARGET, soy_features, alpha=0.0)
soy_pred_direct = predict_nb(soy_model_direct, soy_test[soy_features], use_log=False)

print(f"Soybean direct accuracy (alpha=0): "
      f"{accuracy(soy_test[SOY_TARGET], soy_pred_direct):.2%}")

# Scores of ALL classes are stored before the argmax is taken.
soy_scores_direct = predict_nb_scores(
    soy_model_direct, soy_test[soy_features], use_log=False
)

zero_score_indices = [
    i for i, scores in enumerate(soy_scores_direct)
    if all(v == 0 for v in scores.values())
]

print(f"Number of test samples with all class scores zero: {len(zero_score_indices)} "
      f"(of {len(soy_test)})")
print(f"Zero-score test indices: {zero_score_indices}")

def zero_causing_features(model, row):
    """For every class, the (feature, value) pairs whose conditional probability is exactly 0."""
    result = {}
    for c in model["classes"]:
        bad = []
        for f in model["features"]:
            v = _feature_value(model, f, row[f])
            p = model["cond_probs"][f][c].get(v, 0.0)
            if p == 0.0:
                bad.append((f, row[f]))
        if bad:
            result[c] = bad
    return result

for idx in zero_score_indices[:3]:
    row = soy_test.iloc[idx]
    true_c = row[SOY_TARGET]
    causes = zero_causing_features(soy_model_direct, row)
    never_seen = [f for f in soy_features if row[f] not in set(soy_train[f])]
    print(f"\nZero-score sample index: {idx}, true class: {true_c}")
    print(f"  {len(causes)}/{len(soy_model_direct['classes'])} classes are zeroed; "
          f"fewest zeroing features in one class: {min(len(v) for v in causes.values())}")
    print(f"  test values that never occur anywhere in the training set: {never_seen}")
    shown = [true_c] + [c for c in causes if c != true_c][:2]
    for c in shown:
        pairs = causes[c]
        extra = f" ... (+{len(pairs) - 5} more)" if len(pairs) > 5 else ""
        print(f"  class {c}: {len(pairs)} zero factor(s), e.g. {pairs[:5]}{extra}")

fallback = soy_pred_direct[zero_score_indices[0]] if zero_score_indices else None
fallback_ok = sum(soy_pred_direct[i] == soy_test[SOY_TARGET].iloc[i] for i in zero_score_indices)
print(f"\nFallback prediction (largest prior) for zero-score samples: {fallback!r}")
print(f"Zero-score samples predicted correctly by that fallback: {fallback_ok}/{len(zero_score_indices)}")


Soybean direct accuracy (alpha=0): 82.19%
Number of test samples with all class scores zero: 7 (of 73)
Zero-score test indices: [4, 11, 20, 34, 35, 40, 54]

Zero-score sample index: 4, true class: alternarialeaf-spot
  19/19 classes are zeroed; fewest zeroing features in one class: 1
  test values that never occur anywhere in the training set: []
  class alternarialeaf-spot: 1 zero factor(s), e.g. [('hail', '1.0')]
  class 2-4-d-injury: 34 zero factor(s), e.g. [('date', '6.0'), ('plant-stand', '0.0'), ('precip', '1.0'), ('temp', '1.0'), ('hail', '1.0')] ... (+29 more)
  class anthracnose: 7 zero factor(s), e.g. [('precip', '1.0'), ('leafspots-halo', '2.0'), ('leafspots-marg', '0.0'), ('leafspot-size', '1.0'), ('stem', '0.0')] ... (+2 more)

Zero-score sample index: 11, true class: anthracnose
  19/19 classes are zeroed; fewest zeroing features in one class: 1
  test values that never occur anywhere in the training set: []
  class anthracnose: 1 zero factor(s), e.g. [('date', '1.0')]
  

### Part 5 — Analysis

Without smoothing, the Soybean model reaches only **82.19 %**, and **7 of the 73** test samples (indices 4, 11, 20, 34, 35, 40, 54) have a score of exactly 0 for *all 19 classes*.

**1. Why can the scores of all classes be zero for one sample?**
A class score is the prior times 35 conditional probabilities. If a single factor $P(X_i = v \mid C)$ is 0 – because value $v$ never occurred together with class $C$ in the training set – the whole product is 0, no matter what the other 34 features say. Soybean has 19 classes, 35 features and only 234 training rows (some classes have just a handful, one class a single row), so many (class, value) pairs are never observed. For the zero-score samples every one of the 19 classes contains at least one such zeroing factor. The examples above show it clearly: for sample 4 the *true* class `alternarialeaf-spot` is zeroed by one single feature value (`hail = 1.0`, never seen with that class in training), for sample 11 `anthracnose` by `date = 1.0`, for sample 20 `brown-spot` by `fruit-spots = 1.0`. None of the offending values is new – each occurs in the training set, just never together with that class – so one unlucky rare value is enough to veto the correct class.

**2. How is this different from Underflow?**
Underflow is a *numerical* problem: all factors are positive, the true product is positive but too small for floating point, and working with logarithms solves it. Here the problem is *statistical*: the estimated probability is exactly 0 (the model claims the combination is impossible), so the true score of the model itself is 0 and $\log 0 = -\infty$; logarithms cannot help. Part 4 showed that no real underflow occurs in this data, so all these zeros are zero-probability zeros. The remedy is to change the probability estimates, i.e. smoothing (Part 6).

**3. When all scores are zero, how can the model still choose a class?**
The scores carry no information, so the argmax is a tie between all classes. The implementation breaks ties by the largest prior $P(C)$ and, if priors tie as well, by the first class in alphabetical order. Four classes tie with the maximum of 30 training rows each, so for all 7 samples the model outputs `alternarialeaf-spot` – effectively a guess that ignores the features. That guess is right for only 1 of the 7 samples (sample 4), which is the main reason for the poor 82.19 % accuracy.

## Part 6

In [ ]:
alphas = [0.0, 0.01, 0.05, 0.10]

print(f"{'alpha':>8} | {'direct acc':>12} | {'log acc':>10} | {'zero-score samples':>18}")
print("-" * 60)

soy_results = []

for a in alphas:
    model = fit_nb(soy_train, SOY_TARGET, soy_features, alpha=a)

    scores_direct = predict_nb_scores(model, soy_test[soy_features], use_log=False)
    zero_count = sum(
        all(v == 0 for v in scores.values()) for scores in scores_direct
    )

    pred_log = predict_nb(model, soy_test[soy_features], use_log=True)
    acc_log = accuracy(soy_test[SOY_TARGET], pred_log)

    pred_direct = predict_nb(model, soy_test[soy_features], use_log=False)
    acc_direct = accuracy(soy_test[SOY_TARGET], pred_direct)

    soy_results.append({
        "alpha": a,
        "direct_accuracy": acc_direct,
        "log_accuracy": acc_log,
        "zero_score_samples": zero_count,
    })

    print(f"{a:>8} | {acc_direct:>12.2%} | {acc_log:>10.2%} | {zero_count:>18}")

soy_results_df = pd.DataFrame(soy_results)
soy_results_df

   alpha |   direct acc |    log acc | zero-score samples
------------------------------------------------------------
     0.0 |       82.19% |     82.19% |                  7
    0.01 |       89.04% |     89.04% |                  0
    0.05 |       87.67% |     87.67% |                  0
     0.1 |       84.93% |     84.93% |                  0


,alpha,direct_accuracy,log_accuracy,zero_score_samples
0,0.00,0.821918,0.821918,7
1,0.01,0.890411,0.890411,0
2,0.05,0.876712,0.876712,0
3,0.10,0.849315,0.849315,0


### Part 6 — Analysis

| $\alpha$ | accuracy | correct / 73 | samples with all scores zero |
|---|---|---|---|
| 0 | 82.19 % | 60 | 7 |
| 0.01 | **89.04 %** | 65 | 0 |
| 0.05 | 87.67 % | 64 | 0 |
| 0.10 | 84.93 % | 62 | 0 |

(The ordinary and the logarithmic version give identical accuracies for every $\alpha$.)

**1. Which $\alpha$ gave the best accuracy?** $\alpha = 0.01$ with 89.04 %, which is 6.85 percentage points better than without smoothing.

**2. How does the number of all-zero samples change as $\alpha$ increases?** It drops from 7 at $\alpha = 0$ to 0 for every $\alpha > 0$. With smoothing each conditional probability is $(N(X_i{=}v, C)+\alpha)/(N(C)+\alpha K_i) > 0$ (also for values that never occurred in training, through the `__UNKNOWN__` slot), and a product of positive numbers cannot be zero (and Part 4 showed no underflow here).

**3. Does the largest $\alpha$ necessarily give the best accuracy?** No. A larger $\alpha$ pulls every conditional probability towards the uniform distribution and therefore washes out the evidence in the data: here accuracy *falls* as $\alpha$ grows beyond 0.01 (89.04 % → 87.67 % → 84.93 %). A small $\alpha$ is enough to remove the zeros while changing the observed frequencies as little as possible. The comparison should also be read with care: with 73 test samples one sample is 1.37 percentage points, so 0.01 and 0.05 differ by only one sample, and values between 0 and 0.05 were not tried.

**4. Why is it better to treat $\alpha$ as a tunable parameter?** $\alpha$ cannot be learned by counting (the training likelihood is always highest at $\alpha = 0$), and the right amount of smoothing depends on the data: the number of training rows, classes and feature values, and how sparse or deterministic the counts are. Soybean prefers a small positive $\alpha$, whereas Mushroom prefers almost none (Part 7). So $\alpha$ should be a hyperparameter of the model, chosen on held-out validation data or by cross-validation – ideally not on the test set itself, which would make the reported accuracy optimistic.

## Part 7

In [ ]:
MUSHROOM_ALPHA = 1.0

configs = [
    ("Direct, no smoothing",    False, 0.0),
    ("Log, no smoothing",       True,  0.0),
    ("Direct, with smoothing",  False, MUSHROOM_ALPHA),
    ("Log, with smoothing",     True,  MUSHROOM_ALPHA),
]

print(f"{'configuration':<25} | {'accuracy':>10}")
print("-" * 40)

for name, use_log, alpha in configs:
    model = fit_nb(full_train, TARGET_MUSHROOM, full_features, alpha=alpha)
    preds = predict_nb(model, full_test[full_features], use_log=use_log)
    acc = accuracy(full_test[TARGET_MUSHROOM], preds)
    print(f"{name:<25} | {acc:>10.2%}")

configuration             |   accuracy
----------------------------------------
Direct, no smoothing      |     99.45%
Log, no smoothing         |     99.45%
Direct, with smoothing    |     94.58%
Log, with smoothing       |     94.58%


In [ ]:
m0 = fit_nb(full_train, TARGET_MUSHROOM, full_features, alpha=0.0)
n_zero_mush = sum(all(v == 0 for v in s.values())
                  for s in predict_nb_scores(m0, full_test[full_features], use_log=False))
print(f"Mushroom test samples whose scores are ALL zero without smoothing: {n_zero_mush} / {len(full_test)}")

print(f"\n{'alpha':>8} | {'accuracy':>9} | {'errors':>6}")
print("-" * 30)
for a in [0.0, 0.001, 0.01, 0.1, 0.5, 1.0]:
    m = fit_nb(full_train, TARGET_MUSHROOM, full_features, alpha=a)
    pr = predict_nb(m, full_test[full_features], use_log=True)
    err = sum(p_ != t_ for p_, t_ in zip(pr, full_test[TARGET_MUSHROOM]))
    print(f"{a:>8} | {accuracy(full_test[TARGET_MUSHROOM], pr):>9.2%} | {err:>6}")

Mushroom test samples whose scores are ALL zero without smoothing: 0 / 1625

   alpha |  accuracy | errors
------------------------------
     0.0 |    99.45% |      9
   0.001 |    99.08% |     15
    0.01 |    98.95% |     17
     0.1 |    97.72% |     37
     0.5 |    95.75% |     69
     1.0 |    94.58% |     88


### Part 7 — Analysis

Smoothing parameter used for the smoothed rows: **$\alpha = 1$** (Laplace smoothing).

| Configuration | Test accuracy |
|---|---|
| Direct, no smoothing | 99.45 % |
| Logarithmic, no smoothing | 99.45 % |
| Direct, with smoothing ($\alpha = 1$) | 94.58 % |
| Logarithmic, with smoothing ($\alpha = 1$) | 94.58 % |

**1. Did the accuracy change when smoothing was added?** Yes: it went from 99.45 % to 94.58 %, i.e. 4.87 percentage points lower – 88 instead of 9 misclassified mushrooms out of 1 625.

**2. Did smoothing increase or decrease the accuracy?** It *decreased* it. The log/direct choice makes no difference in either case.

**3. Why can we not say that smoothing increases accuracy on all datasets (Mushroom vs. Soybean)?**
On Soybean smoothing helped a lot (82.19 % → 89.04 % at $\alpha = 0.01$) because the unsmoothed model failed on 7 of 73 test samples through all-zero scores. On Mushroom there is nothing of that kind to repair: with 6 499 training rows and only 2 classes, **none** of the 1 625 test samples has all scores equal to zero without smoothing. Moreover, the dataset is almost deterministic, and here many zero counts are genuine information: e.g. 25 feature values occur at least 50 times in one class and never in the other (such as `odor = f`: 1 754 poisonous, 0 edible). Without smoothing such a value acts as a decisive veto. Smoothing with $\alpha = 1$ replaces "never" by a small positive probability and, since it adds $\alpha K_i$ to the denominator, also flattens all the other estimates, so the strong evidence is diluted and more samples are misclassified. The sweep above agrees: accuracy decreases monotonically as $\alpha$ grows (99.45 % at 0 → 98.95 % at 0.01 → 94.58 % at 1). So whether smoothing helps depends on the amount of data, the number of classes and values, how sparse the counts are, and on the size of $\alpha$ – it is a bias/variance trade-off, not a free improvement.

**4. Role of the logarithm vs. smoothing in one sentence:** the logarithm is a numerical device that leaves the model's probabilities and predictions unchanged and only protects against underflow (it cannot fix a zero probability, since $\log 0 = -\infty$), whereas smoothing changes the probability estimates themselves to avoid zeros, which can change – and sometimes lower – the accuracy.